# ANSC 4040 Mini Project
## 16 GB RAM Optimization: Extra Trees + KNN Benchmark

This notebook is a revised version of the previous Extra Trees workflow.

### Why the old Extra Trees accuracy was extremely low

The previous run had **7,539 training classes**, but the Extra Trees search restricted each tree to roughly **40–80 leaf nodes** and depth **6–8**, while some configurations also used `min_samples_leaf=20`.

That is severe underfitting for a 7,000+ class identification problem. A tree with only dozens of leaves cannot create sufficiently detailed partitions to separate thousands of animals.

The revised workflow:

1. Keeps the chronological train / validation / test design.
2. Adds a highly informative derived feature:
   `LactationStartDate = EventDate - DaysInMilk`.
3. Uses a more realistic Extra Trees search for 16 GB RAM.
4. Adds a **KNN benchmark**, because nearest-neighbor methods can be a better fit for thousands of identity classes and repeated measurements.
5. Uses the *same chronological validation set* for both models, so the comparison is fair.

> Important: no model can honestly be guaranteed to reach 60% accuracy before running it on the data. This notebook is designed to address the main causes of underfitting and to test KNN as an alternative.


In [1]:
# ============================================================
# 1. Imports and settings
# ============================================================

from pathlib import Path
import gc
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

RANDOM_STATE = 42

MODEL_DATA_PATH = Path(
    r"/Library/ansci-4040-fall-2026/jl4937_6040-project-1/dataset/dataset_for_model_buliding_2.csv"
)

PREDICTION_DATA_PATH = Path(
    r"/Library/ansci-4040-fall-2026/jl4937_6040-project-1/dataset/records_missing_AnimalId.csv"
)

# ----------------------------
# 16 GB RAM settings
# ----------------------------

# Extra Trees tuning
ET_ROWS_PER_ANIMAL = 60
ET_VALIDATION_ROWS = 20_000
ET_RANDOM_ITERATIONS = 10
ET_N_JOBS = 2

# KNN tuning
KNN_ROWS_PER_ANIMAL = 60
KNN_VALIDATION_ROWS = 10_000
KNN_MAX_NEIGHBORS = 11
KNN_N_JOBS = -1

# Final evaluation
FINAL_ROWS_PER_ANIMAL = 100
EVALUATION_BATCH_SIZE = 5_000
PREDICTION_BATCH_SIZE = 5_000

RUN_FINAL_PREDICTION = False


In [2]:
# ============================================================
# 2. Load data
# ============================================================

MODEL_COLUMNS = [
    "AnimalId",
    "LactationNumber",
    "DaysInMilk",
    "ReproductionStatus",
    "EventDate",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
    "milking",
]

data = pd.read_csv(
    MODEL_DATA_PATH,
    usecols=MODEL_COLUMNS,
    parse_dates=["EventDate"],
    low_memory=False,
)

integer_cols = [
    "LactationNumber",
    "DaysInMilk",
    "DurationSession_sec",
]

float_cols = [
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
]

for col in integer_cols:
    data[col] = pd.to_numeric(data[col], downcast="integer")

for col in float_cols:
    data[col] = pd.to_numeric(data[col], downcast="float")

for col in ["ReproductionStatus", "milking"]:
    data[col] = data[col].astype("category")

data["AnimalId"] = data["AnimalId"].astype("category")

print(f"Rows: {len(data):,}")
print(f"Known AnimalId classes: {data['AnimalId'].nunique():,}")
print(f"Memory: {data.memory_usage(deep=True).sum()/1024**3:.2f} GB")
display(data.head())


Rows: 6,645,172
Known AnimalId classes: 9,069
Memory: 0.20 GB


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8.839529e+18,1,365,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1,66,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,7.750424e+18,1,244,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
3,5.803650e+18,1,394,Open,2019-12-14,3.492661,4.399846,8.173735,11.793402,198,1
4,-1.713215e+18,5,288,Pregnant,2020-05-11,3.311224,4.300056,7.466130,8.572896,153,2


In [3]:
# ============================================================
# 3. Chronological split
# ============================================================

labelled = data.loc[
    data["AnimalId"].notna()
    & data["EventDate"].notna()
].copy()

labelled = labelled.sort_values("EventDate").reset_index(drop=True)

n = len(labelled)

train_cut = labelled.iloc[int(n * 0.60)]["EventDate"]
validation_cut = labelled.iloc[int(n * 0.80)]["EventDate"]

data_train = labelled.loc[labelled["EventDate"] < train_cut].copy()

data_validation = labelled.loc[
    (labelled["EventDate"] >= train_cut)
    & (labelled["EventDate"] < validation_cut)
].copy()

data_test = labelled.loc[
    labelled["EventDate"] >= validation_cut
].copy()

print(f"Train:      {len(data_train):,}")
print(f"Validation: {len(data_validation):,}")
print(f"Test:       {len(data_test):,}")

print(f"Train classes: {data_train['AnimalId'].nunique():,}")


Train:      3,985,088
Validation: 1,327,822
Test:       1,332,262
Train classes: 7,539


## 4. Feature engineering improvement

The key new feature is:

```text
LactationStartDate = EventDate - DaysInMilk
```

Within the same lactation, this date should be approximately stable for one animal. It can therefore be much more discriminative for identity than using only `EventYear`, `EventMonth`, and `DaysInMilk` separately.

This is derived only from predictors already available at prediction time, so it does not use `AnimalId`.


In [4]:
# ============================================================
# 4. Feature engineering
# ============================================================

def make_features(frame, origin_date):
    X = frame.drop(columns=["AnimalId"], errors="ignore").copy()

    X["EventDate"] = pd.to_datetime(X["EventDate"], errors="coerce")

    # Basic calendar features
    X["EventOrdinal"] = (
        X["EventDate"] - origin_date
    ).dt.days.astype("float32")

    X["EventMonth"] = X["EventDate"].dt.month.astype("float32")
    X["EventDayOfWeek"] = X["EventDate"].dt.dayofweek.astype("float32")

    # Strong identity/lactation feature
    lactation_start = (
        X["EventDate"]
        - pd.to_timedelta(X["DaysInMilk"], unit="D")
    )

    X["LactationStartOrdinal"] = (
        lactation_start - origin_date
    ).dt.days.astype("float32")

    X["LactationStartMonth"] = (
        lactation_start.dt.month.astype("float32")
    )

    X = X.drop(columns=["EventDate"])

    return X


def fit_encoder(frame, origin_date):
    X = make_features(frame, origin_date)

    cat_cols = (
        X.select_dtypes(include=["object", "category"])
        .columns.tolist()
    )

    X = pd.get_dummies(
        X,
        columns=cat_cols,
        dtype=np.float32
    )

    for c in X.columns:
        X[c] = pd.to_numeric(X[c], downcast="float")

    medians = X.median(numeric_only=True)
    X = X.fillna(medians)

    return X.astype(np.float32), cat_cols, X.columns, medians


def transform_encoder(
    frame,
    origin_date,
    cat_cols,
    reference_columns,
    medians
):
    X = make_features(frame, origin_date)

    available_cat_cols = [
        c for c in cat_cols if c in X.columns
    ]

    X = pd.get_dummies(
        X,
        columns=available_cat_cols,
        dtype=np.float32
    )

    X = X.reindex(
        columns=reference_columns,
        fill_value=0
    )

    for c in X.columns:
        X[c] = pd.to_numeric(X[c], downcast="float")

    X = X.fillna(medians)

    return X.astype(np.float32)


In [5]:
# ============================================================
# 5. Target encoding and validation coverage
# ============================================================

class_values = pd.Index(
    data_train["AnimalId"].astype(object).unique()
)

class_to_number = {
    animal_id: i
    for i, animal_id in enumerate(class_values)
}

number_to_class = {
    i: animal_id
    for i, animal_id in enumerate(class_values)
}

validation_target = (
    data_validation["AnimalId"]
    .astype(object)
    .map(class_to_number)
)

known_validation_mask = validation_target.notna()

print(f"Train classes: {len(class_values):,}")
print(
    "Validation coverage:",
    f"{known_validation_mask.mean():.2%}"
)


Train classes: 7,539
Validation coverage: 91.95%


In [6]:
# ============================================================
# 6. Class-aware sample helper
# ============================================================

def sample_per_class(
    frame,
    max_rows_per_class,
    random_state=42
):
    pieces = []

    for _, group in frame.groupby(
        "AnimalId",
        observed=True,
        sort=False
    ):
        n_take = min(
            len(group),
            max_rows_per_class
        )

        pieces.append(
            group.sample(
                n=n_take,
                random_state=random_state
            )
        )

    return pd.concat(pieces, axis=0)


# Part A — Improved Extra Trees

The previous model was too shallow. This search allows substantially more detailed trees, while still capping tree size enough to protect 16 GB RAM.

Because there are thousands of classes, this model may still be memory-inefficient compared with KNN.


In [7]:
# ============================================================
# 7. Extra Trees tuning sample
# ============================================================

et_train_raw = sample_per_class(
    data_train,
    max_rows_per_class=ET_ROWS_PER_ANIMAL,
    random_state=RANDOM_STATE
)

origin_date = data_train["EventDate"].min()

(
    X_et_train,
    et_cat_cols,
    et_reference_columns,
    et_medians
) = fit_encoder(
    et_train_raw,
    origin_date
)

y_et_train = (
    et_train_raw["AnimalId"]
    .astype(object)
    .map(class_to_number)
    .astype(np.int32)
)

validation_seen = data_validation.loc[
    known_validation_mask
]

et_val_raw = validation_seen.sample(
    n=min(ET_VALIDATION_ROWS, len(validation_seen)),
    random_state=RANDOM_STATE
)

X_et_val = transform_encoder(
    et_val_raw,
    origin_date,
    et_cat_cols,
    et_reference_columns,
    et_medians
)

y_et_val = (
    et_val_raw["AnimalId"]
    .astype(object)
    .map(class_to_number)
    .astype(np.int32)
)

print("ET train shape:", X_et_train.shape)
print("ET validation shape:", X_et_val.shape)


ET train shape: (435179, 19)
ET validation shape: (20000, 19)


In [8]:
# ============================================================
# 8. More realistic Extra Trees random search
# ============================================================

# Major differences from old search:
# - deeper trees
# - many more leaf nodes
# - min_samples_leaf close to 1
# - more trees

et_parameter_space = {
    "n_estimators": [60, 80, 100, 120],
    "max_depth": [16, 20, 24, 28, None],
    "max_leaf_nodes": [512, 1024, 2048, 4096],
    "min_samples_split": [2, 4, 8],
    "min_samples_leaf": [1, 2, 3],
    "max_features": ["sqrt", 0.5, 0.8, 1.0],
    "criterion": ["gini", "entropy"],
    "bootstrap": [False],
    "class_weight": [None, "balanced"],
}

et_parameter_sets = list(
    ParameterSampler(
        et_parameter_space,
        n_iter=ET_RANDOM_ITERATIONS,
        random_state=RANDOM_STATE
    )
)

et_results = []
best_et_model = None
best_et_params = None
best_et_accuracy = -1

for i, params in enumerate(et_parameter_sets, start=1):
    print("\n" + "=" * 70)
    print(f"Extra Trees configuration {i}/{len(et_parameter_sets)}")
    print(params)

    model = ExtraTreesClassifier(
        **params,
        n_jobs=ET_N_JOBS,
        random_state=RANDOM_STATE
    )

    start = time.time()

    try:
        model.fit(X_et_train, y_et_train)

        pred = model.predict(X_et_val)

        acc = accuracy_score(y_et_val, pred)
        bal = balanced_accuracy_score(y_et_val, pred)
        wf1 = f1_score(
            y_et_val,
            pred,
            average="weighted",
            zero_division=0
        )

        elapsed = (time.time() - start) / 60

        et_results.append({
            **params,
            "accuracy": acc,
            "balanced_accuracy": bal,
            "weighted_f1": wf1,
            "minutes": elapsed
        })

        print(f"Accuracy: {acc:.4f}")
        print(f"Balanced accuracy: {bal:.4f}")
        print(f"Weighted F1: {wf1:.4f}")
        print(f"Minutes: {elapsed:.2f}")

        if acc > best_et_accuracy:
            best_et_accuracy = acc
            best_et_model = model
            best_et_params = params.copy()

    except MemoryError:
        print(
            "MemoryError: this configuration exceeded RAM. "
            "It was skipped."
        )

    gc.collect()

et_results_df = (
    pd.DataFrame(et_results)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

display(et_results_df)

print("\nBest Extra Trees accuracy:", best_et_accuracy)
print("Best Extra Trees params:", best_et_params)



Extra Trees configuration 1/10
{'n_estimators': 100, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_leaf_nodes': 1024, 'max_features': 0.8, 'max_depth': 24, 'criterion': 'gini', 'class_weight': 'balanced', 'bootstrap': False}


/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Accuracy: 0.1287
Balanced accuracy: 0.1372
Weighted F1: 0.0913
Minutes: 0.95

Extra Trees configuration 2/10
{'n_estimators': 60, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_leaf_nodes': 4096, 'max_features': 0.5, 'max_depth': 20, 'criterion': 'gini', 'class_weight': None, 'bootstrap': False}


/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Accuracy: 0.2606
Balanced accuracy: 0.2652
Weighted F1: 0.2443
Minutes: 0.83

Extra Trees configuration 3/10
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_leaf_nodes': 1024, 'max_features': 0.5, 'max_depth': None, 'criterion': 'entropy', 'class_weight': None, 'bootstrap': False}


...

/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Accuracy: 0.2486
Balanced accuracy: 0.2531
Weighted F1: 0.2178
Minutes: 1.33

Extra Trees configuration 10/10
{'n_estimators': 100, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_leaf_nodes': 2048, 'max_features': 0.8, 'max_depth': None, 'criterion': 'entropy', 'class_weight': None, 'bootstrap': False}
Accuracy: 0.2486
Balanced accuracy: 0.2531
Weighted F1: 0.2178
Minutes: 1.33


/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,n_estimators,min_samples_split,min_samples_leaf,max_leaf_nodes,max_features,max_depth,criterion,class_weight,bootstrap,accuracy,balanced_accuracy,weighted_f1,minutes
0,100,8,1,4096,1.0,NaN,entropy,NaN,False,0.29225,0.286473,0.253005,2.009934
1,60,8,3,4096,0.5,20.0,gini,NaN,False,0.26060,0.265189,0.244324,0.825560
2,100,8,3,2048,0.8,24.0,entropy,NaN,False,0.24860,0.253125,0.217832,1.328971
3,100,8,3,2048,0.8,NaN,entropy,NaN,False,0.24860,0.253125,0.217832,1.328534
4,60,4,2,1024,0.8,NaN,entropy,balanced,False,0.20425,0.210674,0.171361,0.648372
5,100,2,3,1024,0.5,NaN,entropy,NaN,False,0.16945,0.178303,0.146858,0.797137
6,80,2,1,2048,1.0,16.0,gini,balanced,False,0.14985,0.159783,0.108219,1.212360
7,100,8,3,1024,0.8,24.0,gini,balanced,False,0.12870,0.137231,0.091262,0.946346
8,100,8,3,512,1.0,16.0,gini,NaN,False,0.07505,0.078970,0.039745,0.931433
9,120,4,1,512,sqrt,NaN,entropy,NaN,False,0.05885,0.063704,0.046180,0.530525



Best Extra Trees accuracy: 0.29225
Best Extra Trees params: {'n_estimators': 100, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_leaf_nodes': 4096, 'max_features': 1.0, 'max_depth': None, 'criterion': 'entropy', 'class_weight': None, 'bootstrap': False}


# Part B — KNN benchmark

KNN can be well suited to this problem because:

- there are thousands of target classes,
- each cow has repeated measurements,
- KNN does not need to build a huge multiclass probability vector at every tree node,
- the derived lactation-start feature can make records from the same cow/lactation close in feature space.

To make tuning efficient, nearest neighbors are calculated once for each distance metric, and several values of `k` and weighting rules are evaluated from the same neighbor search.


In [9]:
# ============================================================
# 9. Prepare KNN data
# ============================================================

knn_train_raw = sample_per_class(
    data_train,
    max_rows_per_class=KNN_ROWS_PER_ANIMAL,
    random_state=RANDOM_STATE
)

(
    X_knn_train_df,
    knn_cat_cols,
    knn_reference_columns,
    knn_medians
) = fit_encoder(
    knn_train_raw,
    origin_date
)

y_knn_train = (
    knn_train_raw["AnimalId"]
    .astype(object)
    .map(class_to_number)
    .astype(np.int32)
    .to_numpy()
)

knn_val_raw = validation_seen.sample(
    n=min(KNN_VALIDATION_ROWS, len(validation_seen)),
    random_state=RANDOM_STATE
)

X_knn_val_df = transform_encoder(
    knn_val_raw,
    origin_date,
    knn_cat_cols,
    knn_reference_columns,
    knn_medians
)

y_knn_val = (
    knn_val_raw["AnimalId"]
    .astype(object)
    .map(class_to_number)
    .astype(np.int32)
    .to_numpy()
)

# Standardization is essential for distance-based models.
scaler = StandardScaler()

X_knn_train = scaler.fit_transform(
    X_knn_train_df
).astype(np.float32)

X_knn_val = scaler.transform(
    X_knn_val_df
).astype(np.float32)

print("KNN train shape:", X_knn_train.shape)
print("KNN validation shape:", X_knn_val.shape)


KNN train shape: (435179, 19)
KNN validation shape: (10000, 19)


In [10]:
# ============================================================
# 10. KNN tuning utilities
# ============================================================

def vote_from_neighbors(
    neighbor_labels,
    neighbor_distances,
    k,
    weights="distance"
):
    predictions = np.empty(
        len(neighbor_labels),
        dtype=np.int32
    )

    for i in range(len(neighbor_labels)):
        labels = neighbor_labels[i, :k]
        distances = neighbor_distances[i, :k]

        unique_labels = np.unique(labels)

        best_label = None
        best_score = -1

        for label in unique_labels:
            mask = labels == label

            if weights == "distance":
                score = np.sum(
                    1.0 / (distances[mask] + 1e-8)
                )
            else:
                score = np.sum(mask)

            if score > best_score:
                best_score = score
                best_label = label

        predictions[i] = best_label

    return predictions


In [11]:
# ============================================================
# 11. KNN hyperparameter search
# ============================================================

knn_results = []

best_knn_accuracy = -1
best_knn_config = None
best_knn_search_model = None

for metric, p in [
    ("euclidean", 2),
    ("manhattan", 1),
]:
    print("\n" + "=" * 70)
    print("Neighbor search metric:", metric)

    nn = NearestNeighbors(
        n_neighbors=KNN_MAX_NEIGHBORS,
        algorithm="auto",
        metric="minkowski",
        p=p,
        n_jobs=KNN_N_JOBS
    )

    start = time.time()

    nn.fit(X_knn_train)

    distances, indices = nn.kneighbors(
        X_knn_val,
        return_distance=True
    )

    neighbor_labels = y_knn_train[indices]

    search_minutes = (
        time.time() - start
    ) / 60

    for k in [1, 3, 5, 7, 11]:
        for weights in ["uniform", "distance"]:

            pred = vote_from_neighbors(
                neighbor_labels,
                distances,
                k=k,
                weights=weights
            )

            acc = accuracy_score(
                y_knn_val,
                pred
            )

            bal = balanced_accuracy_score(
                y_knn_val,
                pred
            )

            wf1 = f1_score(
                y_knn_val,
                pred,
                average="weighted",
                zero_division=0
            )

            result = {
                "metric": metric,
                "k": k,
                "weights": weights,
                "accuracy": acc,
                "balanced_accuracy": bal,
                "weighted_f1": wf1,
                "neighbor_search_minutes": search_minutes,
            }

            knn_results.append(result)

            print(
                f"{metric}, k={k}, {weights}: "
                f"accuracy={acc:.4f}"
            )

            if acc > best_knn_accuracy:
                best_knn_accuracy = acc
                best_knn_config = {
                    "metric": metric,
                    "p": p,
                    "k": k,
                    "weights": weights
                }

knn_results_df = (
    pd.DataFrame(knn_results)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

display(knn_results_df)

print("\nBest KNN validation accuracy:", best_knn_accuracy)
print("Best KNN configuration:", best_knn_config)



Neighbor search metric: euclidean


/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


euclidean, k=1, uniform: accuracy=0.0150
euclidean, k=1, distance: accuracy=0.0150
euclidean, k=3, uniform: accuracy=0.0100


/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


euclidean, k=3, distance: accuracy=0.0151
euclidean, k=5, uniform: accuracy=0.0093


...

/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


manhattan, k=7, distance: accuracy=0.0275
manhattan, k=11, uniform: accuracy=0.0188
manhattan, k=11, distance: accuracy=0.0255


/Users/robinli/.pyenv/versions/3.13.7/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,metric,k,weights,accuracy,balanced_accuracy,weighted_f1,neighbor_search_minutes
0,manhattan,5,distance,0.0286,0.031610,0.029113,0.085513
1,manhattan,7,distance,0.0275,0.030492,0.027509,0.085513
2,manhattan,3,distance,0.0272,0.030522,0.027580,0.085513
3,manhattan,1,uniform,0.0267,0.029753,0.026930,0.085513
4,manhattan,1,distance,0.0267,0.029753,0.026930,0.085513
5,manhattan,11,distance,0.0255,0.028751,0.025498,0.085513
6,manhattan,7,uniform,0.0209,0.023219,0.021260,0.085513
7,manhattan,5,uniform,0.0205,0.023303,0.020406,0.085513
8,manhattan,3,uniform,0.0204,0.022747,0.020174,0.085513
9,manhattan,11,uniform,0.0188,0.020931,0.019094,0.085513



Best KNN validation accuracy: 0.0286
Best KNN configuration: {'metric': 'manhattan', 'p': 1, 'k': 5, 'weights': 'distance'}


In [12]:
# ============================================================
# 12. Compare Extra Trees vs KNN
# ============================================================

comparison = pd.DataFrame([
    {
        "model": "Extra Trees",
        "validation_accuracy": best_et_accuracy
    },
    {
        "model": "KNN",
        "validation_accuracy": best_knn_accuracy
    }
]).sort_values(
    "validation_accuracy",
    ascending=False
)

display(comparison)

selected_model_name = comparison.iloc[0]["model"]

print("Selected model based on validation accuracy:")
print(selected_model_name)


,model,validation_accuracy
0,Extra Trees,0.29225
1,KNN,0.02860


Selected model based on validation accuracy:
Extra Trees


## How to interpret the result

If KNN reaches or approaches 60% while Extra Trees remains low, that is evidence that the identity problem is better represented as a **local similarity / nearest-neighbor problem** than as a large multiclass tree problem.

If both remain far below 60%, check whether your classmate used the same evaluation design. A random row split can be much easier than a chronological split because records from the same cow and nearby dates may appear in both train and test sets.

For a fair comparison, the split strategy must be the same.


In [13]:
# ============================================================
# 13. Optional diagnostic:
# compare with a RANDOM stratified split on a small sample
# ============================================================

# This cell is intentionally not executed automatically.
#
# If a classmate reports ~60% accuracy, ask whether they used a
# random split rather than chronological validation.
#
# A random split can produce much higher accuracy because repeated
# measurements from the same cow are distributed across both sets.
#
# Do NOT replace the chronological evaluation unless your project
# requirements justify a random split.


# Next step

Use the validation comparison above first.

- If **KNN is clearly better**, continue the project with KNN.
- If **Extra Trees improves substantially**, you can refine its search around the best configuration.
- Do not tune using the held-out test period.
- Only after the final model choice should the test period be evaluated once.
